# Notebook de Test - Pipeline d'Ingestion SFTP

Ce notebook permet de tester le pipeline d'ingestion des données via SFTP en suivant les mêmes étapes que `sftp_ingestion_pipeline()` :
1. **Configuration SFTP** - Chargement de la configuration
2. **Listing des fichiers** - Lister les fichiers disponibles sur le serveur
3. **Traitement des fichiers** - Téléchargement, parsing, matching, mise à jour BD, archivage
4. **Résumé** - Statistiques de l'exécution

## 0. Initialisation et Configuration

In [ ]:
import os
import logging
import warnings
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Ignorer certains warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

print("✅ Initialisation terminée")

✅ Initialisation terminée


In [ ]:
# Charger la configuration SFTP depuis les variables d'environnement
sftp_config = {
    'enabled': os.getenv('SFTP_ENABLED', 'false').lower() == 'true',
    'host': os.getenv('SFTP_HOST'),
    'port': int(os.getenv('SFTP_PORT', '22')),
    'username': os.getenv('SFTP_USERNAME'),
    'ssh_private_key_b64': os.getenv('SFTP_PRIVATE_KEY_B64'),
    'ssh_private_key_content': os.getenv('SFTP_SSH_PRIVATE_KEY_CONTENT'),
    'passphrase': os.getenv('SFTP_PASSPHRASE'),
    'timeout': int(os.getenv('SFTP_TIMEOUT', '30')),
    'remote_directory': os.getenv('SFTP_REMOTE_DIRECTORY', '/data/incoming'),
    'archive_directory': os.getenv('SFTP_ARCHIVE_DIRECTORY', '/data/archived'),
    'file_pattern': os.getenv('SFTP_FILE_PATTERN', '*.csv'),
    'temp_local_dir': os.getenv('SFTP_TEMP_LOCAL_DIR', '/tmp/sftp_temp')
}

print("✅ Configuration SFTP chargée depuis les variables d'environnement")
print(f"Enabled: {sftp_config.get('enabled')}")
print(f"Host: {sftp_config.get('host')}")
print(f"Username: {sftp_config.get('username')}")

✅ Configuration SFTP chargée depuis les variables d'environnement
Enabled: False
Host: 185.31.41.85
Username: lunarte_sge


In [ ]:
print('=== CONFIGURATION SFTP ===')
print(f"Enabled: {sftp_config.get('enabled', False)}")
print(f"Host: {sftp_config.get('host')}")
print(f"Username: {sftp_config.get('username')}")
print(f"Port: {sftp_config.get('port', 22)}")
print(f"Remote directory: {sftp_config.get('remote_directory')}")
print(f"Archive directory: {sftp_config.get('archive_directory')}")
print(f"File pattern: {sftp_config.get('file_pattern')}")
print(f"SSH private key b64: {'***' if sftp_config.get('ssh_private_key_b64') else 'Not set'}")
print(f"SSH private key content: {'***' if sftp_config.get('ssh_private_key_content') else 'Not set'}")

=== CONFIGURATION SFTP ===
Enabled: False
Host: 185.31.41.85
Username: lunarte_sge
Port: 22
Remote directory: /home/lunarte/sge
Archive directory: /home/lunarte/sge/archived
File pattern: *.csv
SSH private key b64: ***
SSH private key content: Not set


In [ ]:
# Vérifier que la configuration SFTP est présente
if not sftp_config.get('host') or not sftp_config.get('username') or not (sftp_config.get('ssh_private_key_b64') or sftp_config.get('ssh_private_key_content')):
    print("❌ Configuration SFTP manquante dans les variables d'environnement")
    print("→ Configurez SFTP_HOST, SFTP_USERNAME et SFTP_PRIVATE_KEY_B64 ou SFTP_SSH_PRIVATE_KEY_CONTENT dans .env.secrets")
else:
    print("✅ Configuration SFTP présente")

    # Vérifier les champs requis
    missing = []
    if not sftp_config.get('host'):
        missing.append('host')
    if not sftp_config.get('username'):
        missing.append('username')
    if not (sftp_config.get('ssh_private_key_b64') or sftp_config.get('ssh_private_key_content')):
        missing.append('ssh_private_key_b64|ssh_private_key_content')
    if not sftp_config.get('remote_directory'):
        missing.append('remote_directory')

    if missing:
        print(f"⚠️ Champs requis manquants: {missing}")
    else:
        print("✅ Tous les champs requis sont présents")


✅ Configuration SFTP présente
✅ Tous les champs requis sont présents


In [ ]:
if sftp_config.get('host') and sftp_config.get('username') and (sftp_config.get('ssh_private_key_b64') or sftp_config.get('ssh_private_key_content')):
    try:
        from ml.pipelines.sftp_ingestion_pipeline import run_sftp_ingestion_pipeline
        from ml.config import load_config

        db_uri = os.getenv('PREDICTIONS_POSTGRES_URI')
        if not db_uri:
            config = load_config()
            db_uri = config.get('database', {}).get('uri')
        print(f"DB URI présent: {'oui' if db_uri else 'non'}")

        result = run_sftp_ingestion_pipeline(
            sftp_host=sftp_config['host'],
            sftp_username=sftp_config['username'],
            ssh_private_key_b64=sftp_config.get('ssh_private_key_b64'),
            ssh_private_key_content=sftp_config.get('ssh_private_key_content'),
            db_uri=db_uri,
            passphrase=sftp_config.get('passphrase'),
            remote_directory=sftp_config.get('remote_directory'),
            archive_directory=sftp_config.get('archive_directory'),
            sftp_port=sftp_config.get('port', 22),
            sftp_timeout=sftp_config.get('timeout', 30),
            file_pattern=sftp_config.get('file_pattern', '*.csv'),
            temp_local_dir=sftp_config.get('temp_local_dir', '/tmp/sftp_temp')
        )

        print("✅ Pipeline métier SFTP exécutée")
        print(f"Status: {result.get('status')}")
        print(f"Files count: {result.get('files_count')}")

        if result.get('summary'):
            summary = result['summary']
            print("📊 Résumé de l'exécution :")
            print(f"  Total fichiers: {summary.get('total_files', 0)}")
            print(f"  Réussis: {summary.get('successful', 0)}")
            print(f"  Échoués: {summary.get('failed', 0)}")
            print(f"  Enregistrements traités: {summary.get('total_records_processed', 0)}")
            print(f"  Prédictions mises à jour: {summary.get('total_predictions_updated', 0)}")
            print(f"  Taux de succès: {summary.get('success_rate', 0):.2%}")

    except Exception as e:
        print(f"❌ Erreur lors de l'exécution du pipeline SFTP métier: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Configuration SFTP manquante, test de pipeline ignoré")

2026-06-13 11:36:30,517 - ml.connectors.sftp.sftp_connector - INFO - Clé SSH chargée depuis ssh_private_key_b64


DB URI présent: oui


2026-06-13 11:36:30,520 - ml.connectors.sftp.sftp_data_processor - INFO - Processeur de données SFTP initialisé
2026-06-13 11:36:30,522 - ml.connectors.sftp.sftp_connector - INFO - Échec via PKey.from_private_key: PKey.__init__() got an unexpected keyword argument 'file_obj'
2026-06-13 11:36:30,922 - ml.connectors.sftp.sftp_connector - INFO - Échec via RSAKey.from_private_key_file: unpack requires a buffer of 4 bytes
2026-06-13 11:36:31,346 - ml.connectors.sftp.sftp_connector - INFO - Échec via ECDSAKey.from_private_key_file: 'utf-8' codec can't decode byte 0xc2 in position 0: invalid continuation byte
2026-06-13 11:36:31,611 - ml.connectors.sftp.sftp_connector - INFO - Clé chargée avec succès via Ed25519Key.from_private_key_file
2026-06-13 11:36:31,664 - paramiko.transport - INFO - Connected (version 2.0, client OpenSSH_9.2p1)
2026-06-13 11:36:31,838 - paramiko.transport - INFO - Authentication (publickey) successful!
2026-06-13 11:36:32,102 - paramiko.transport.sftp - INFO - [chan 0]

✅ Pipeline métier SFTP exécutée
Status: success
Files count: 2
📊 Résumé de l'exécution :
  Total fichiers: 2
  Réussis: 1
  Échoués: 1
  Enregistrements traités: 98
  Prédictions mises à jour: 72
  Taux de succès: 50.00%


## ÉTAPE 4: Résumé

## Vérification des Mises à Jour dans la Base de Données

In [ ]:
if sftp_config.get('host') and sftp_config.get('username') and (sftp_config.get('ssh_private_key_b64') or sftp_config.get('ssh_private_key_content')):
    try:
        from ml.pipelines.database_handler import DatabaseHandler

        db_uri = os.getenv('PREDICTIONS_POSTGRES_URI')
        db_handler = DatabaseHandler(db_uri)

        # Récupérer les statistiques
        stats = db_handler.get_prediction_stats()

        if stats:
            print("📊 Statistiques de la base de données:")
            print(f"  Total prédictions: {stats.get('total_predictions', 0)}")
            print(f"  Table existe: {stats.get('table_exists', False)}")

        # Récupérer les prédictions avec valeurs réelles
        from datetime import datetime, timedelta
        today = datetime.now().date()
        yesterday = today - timedelta(days=1)

        predictions_with_actuals = db_handler.get_predictions_by_date(
            start_date=yesterday,
            end_date=today
        )

        if predictions_with_actuals is not None and not predictions_with_actuals.empty:
            with_actuals = predictions_with_actuals[predictions_with_actuals['actual_value'].notna()]
            print(f"\nPrédictions avec valeurs réelles (hier): {len(with_actuals)}")

            if len(with_actuals) > 0:
                display(with_actuals[['prediction_date', 'prediction', 'actual_value']].head(10))
        else:
            print("⚠️ Aucune prédiction trouvée pour la période")

    except Exception as e:
        print(f"❌ Erreur lors de la vérification: {e}")
else:
    print("⚠️ Configuration SFTP manquante, vérification ignorée")

📊 Statistiques de la base de données:
  Total prédictions: 216
  Table existe: True

Prédictions avec valeurs réelles (hier): 3


,prediction_date,prediction,actual_value
0,2026-06-13,536.40625,452.0
1,2026-06-13,536.40625,452.0
2,2026-06-13,536.40625,452.0


## Résumé

Ce notebook a démontré l'exécution du pipeline d'ingestion SFTP en suivant les mêmes étapes que `sftp_ingestion_pipeline()` :
1. ✅ **Configuration SFTP** - Chargement et vérification de la configuration
2. ✅ **Listing des fichiers** - Liste des fichiers disponibles sur le serveur SFTP
3. ✅ **Traitement des fichiers** - Téléchargement, parsing, matching, mise à jour BD, archivage
4. ✅ **Résumé** - Statistiques de l'exécution
5. ✅ **Vérification BD** - Vérification des mises à jour dans la base de données